# PATHQ — Week 5: Hybrid VQC+GNN Training & Ablations

**Goal:** Train the full quantum-hybrid model and prove it beats the classical baseline.

**By end of this notebook:**
- Hybrid VQC+GNN trained on PatchCamelyon
- AUC comparison table: Classical vs Quantum
- Ablation: 1 vs 2 vs 3 VQC layers
- Low-data experiment: 20% / 50% / 100% of training data
- Final comparison figure for your paper

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
import pennylane as qml
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGLoader
from torch_geometric.nn import GCNConv
from sklearn.metrics import roc_auc_score, f1_score
from scipy.spatial import KDTree
from datasets import load_dataset
from torchvision import transforms
import timm, wandb, json, warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42)

ROOT     = Path('..')
CKPT_DIR = ROOT / 'checkpoints'
OUT_DIR  = ROOT / 'outputs'
CKPT_DIR.mkdir(exist_ok=True); OUT_DIR.mkdir(exist_ok=True)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

/home/kabilash/miniconda3/envs/pathq/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## Cell 2 — Full PATHQ Hybrid Model

In [3]:
# ── VQC Encoder ──────────────────────────────────────────────────────────
class VQCEncoder(nn.Module):
    def __init__(self, in_dim=512, n_qubits=3, n_layers=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.vqc_dim  = 2 ** n_qubits
        self.proj = nn.Sequential(nn.Linear(in_dim, self.vqc_dim), nn.Tanh())
        dev = qml.device('default.qubit', wires=n_qubits)

        @qml.qnode(dev, interface='torch', diff_method='parameter-shift')
        def circuit(inputs, weights):
            qml.AmplitudeEmbedding(inputs, wires=range(n_qubits), normalize=True)
            for l in range(n_layers):
                for q in range(n_qubits): qml.RY(weights[l,0,q], wires=q)
                for q in range(n_qubits): qml.RZ(weights[l,1,q], wires=q)
                for q in range(n_qubits-1): qml.CNOT(wires=[q,q+1])
                qml.CNOT(wires=[n_qubits-1,0])
            return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]

        self.vqc = qml.qnn.TorchLayer(circuit, {'weights': (n_layers, 2, n_qubits)})
        self.out_dim = self.vqc_dim + n_qubits

    def forward(self, x):
        p = self.proj(x)
        return torch.cat([p, self.vqc(F.normalize(p, p=2, dim=1))], dim=1)


# ── ABMIL Attention ───────────────────────────────────────────────────────
class ABMILAttention(nn.Module):
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.V = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh())
        self.U = nn.Sequential(nn.Linear(dim, hidden), nn.Sigmoid())
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, x, batch_idx):
        A = self.w(self.V(x) * self.U(x))
        B = batch_idx.max().item() + 1
        feats, attn = [], torch.zeros_like(A)
        for b in range(B):
            m = (batch_idx == b)
            w = torch.softmax(A[m], 0); attn[m] = w
            feats.append((w * x[m]).sum(0, keepdim=True))
        return torch.cat(feats, 0), attn


# ── GNN Encoder ───────────────────────────────────────────────────────────
class GNNEncoder(nn.Module):
    def __init__(self, in_dim, hidden=256, dropout=0.3):
        super().__init__()
        self.proj  = nn.Sequential(nn.Linear(in_dim, hidden), nn.LayerNorm(hidden),
                                    nn.ReLU(), nn.Dropout(dropout))
        self.conv1 = GCNConv(hidden, hidden); self.bn1 = nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden); self.bn2 = nn.BatchNorm1d(hidden)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, ei):
        x = self.proj(x)
        x = x + self.drop(F.relu(self.bn1(self.conv1(x, ei))))
        x = x + F.relu(self.bn2(self.conv2(x, ei)))
        return x


# ── Full PATHQ model ──────────────────────────────────────────────────────
class PATHQHybrid(nn.Module):
    """
    Full PATHQ: VQCEncoder -> GNNEncoder -> ABMIL -> Classifier.
    Set use_vqc=False for classical baseline (replicates Week 3 model).
    """
    def __init__(self, in_dim=512, hidden=256, n_qubits=3, n_layers=2,
                 n_classes=2, dropout=0.3, use_vqc=True):
        super().__init__()
        self.use_vqc = use_vqc
        if use_vqc:
            self.vqc  = VQCEncoder(in_dim, n_qubits, n_layers)
            gnn_in    = self.vqc.out_dim           # 11
        else:
            gnn_in    = in_dim                     # 512
        self.gnn  = GNNEncoder(gnn_in, hidden, dropout)
        self.attn = ABMILAttention(hidden, hidden//2)
        self.head = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        if self.use_vqc: x = self.vqc(x)
        x  = self.gnn(x, ei)
        sf, attn = self.attn(x, b)
        return self.head(sf), attn


# Test both modes
fake = Data(x=torch.randn(32,512), edge_index=torch.randint(0,32,(2,128)),
            y=torch.tensor([1]), batch=torch.zeros(32,dtype=torch.long)).to(DEVICE)

classical = PATHQHybrid(use_vqc=False).to(DEVICE)
quantum   = PATHQHybrid(use_vqc=True).to(DEVICE)

with torch.no_grad():
    lc, ac = classical(fake)
    lq, aq = quantum(fake)

print(f'Classical: logits {lc.shape}  attn {ac.shape}')
print(f'Quantum:   logits {lq.shape}  attn {aq.shape}')
print('Both models working.')

Classical: logits torch.Size([1, 2])  attn torch.Size([32, 1])
Quantum:   logits torch.Size([1, 2])  attn torch.Size([32, 1])
Both models working.


## Cell 3 — Training infrastructure (shared functions)

In [4]:
def train_one(model, loader, opt, device):
    model.train(); total, n = 0., 0
    for batch in loader:
        batch = batch.to(device); opt.zero_grad()
        logits, _ = model(batch)
        loss = F.cross_entropy(logits, batch.y.squeeze())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); total += loss.item(); n += 1
    return total / max(n, 1)

@torch.no_grad()
def eval_model(model, loader, device):
    model.eval(); probs, labels, tl, n = [], [], 0., 0
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch)
        tl += F.cross_entropy(logits, batch.y.squeeze()).item()
        probs.extend(torch.softmax(logits,1)[:,1].cpu().tolist())
        labels.extend(batch.y.squeeze().cpu().tolist()); n += 1
    p, l = np.array(probs), np.array(labels)
    preds = (p >= 0.5).astype(int)
    auc  = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    f1   = f1_score(l, preds, zero_division=0)
    tp, fn = ((preds==1)&(l==1)).sum(), ((preds==0)&(l==1)).sum()
    tn, fp = ((preds==0)&(l==0)).sum(), ((preds==1)&(l==0)).sum()
    return {'auc': auc, 'f1': f1, 'loss': tl/max(n,1),
            'sensitivity': tp/max(tp+fn,1), 'specificity': tn/max(tn+fp,1)}


def run_experiment(model, tr, va, te, device, label,
                   epochs=40, lr=1e-4, ckpt=None, use_wandb=True):
    opt   = torch.optim.Adam(filter(lambda p:p.requires_grad, model.parameters()),
                              lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    if use_wandb: wandb.init(project='PATHQ', name=label, reinit=True,
                              config={'model':label,'epochs':epochs,'lr':lr})
    best_auc, history = 0., []
    for ep in range(1, epochs+1):
        tl = train_one(model, tr, opt, device)
        vm = eval_model(model, va, device)
        sched.step()
        history.append({'ep':ep,'train_loss':tl,'val_auc':vm['auc']})
        if use_wandb: wandb.log({'epoch':ep,'train/loss':tl,'val/auc':vm['auc'],'val/f1':vm['f1']})
        if ep % 10 == 0 or ep == 1:
            print(f'  [{label}] Ep {ep:3d}  loss={tl:.4f}  auc={vm["auc"]:.4f}')
        if vm['auc'] > best_auc:
            best_auc = vm['auc']
            if ckpt: torch.save({'model_state':model.state_dict(),'auc':best_auc}, ckpt)
    if use_wandb: wandb.finish()
    # Test with best weights
    if ckpt and Path(ckpt).exists():
        model.load_state_dict(torch.load(ckpt, weights_only=False)['model_state'])
    tm = eval_model(model, te, device)
    print(f'  [{label}] TEST  auc={tm["auc"]:.4f}  f1={tm["f1"]:.4f}  '
          f'sens={tm["sensitivity"]:.3f}  spec={tm["specificity"]:.3f}')
    return tm, history


print('Training functions ready.')

Training functions ready.


## Cell 4 — Build PatchCamelyon MIL bags (same as Week 3)

In [5]:
BAG_SIZE = 32
print('Loading PatchCamelyon...')
pcam = load_dataset('1aurent/PatchCamelyon')

backbone  = timm.create_model('resnet50', pretrained=True, num_classes=0)
extractor = backbone.to(DEVICE).eval()
for p in extractor.parameters(): p.requires_grad = False

tfm = transforms.Compose([
    transforms.Resize((96,96)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def make_graphs(split, n_bags=500, seed=42):
    rng  = np.random.default_rng(seed)
    ds   = pcam[split]
    idxs = rng.permutation(len(ds))[:n_bags*BAG_SIZE]
    graphs = []
    for i in range(0, len(idxs), BAG_SIZE):
        bi = idxs[i:i+BAG_SIZE]
        if len(bi) < BAG_SIZE: continue
        imgs = torch.stack([tfm(ds[int(j)]['image']) for j in bi]).to(DEVICE)
        lbl  = 1 if any(ds[int(j)]['label'] for j in bi) else 0
        with torch.no_grad():
            feats = extractor(imgs).cpu()
        side = int(BAG_SIZE**0.5)
        coords = torch.tensor([(r,c) for r in range(side) for c in range(side)],
                               dtype=torch.float32)[:BAG_SIZE]
        tree = KDTree(coords.numpy()); _, nn = tree.query(coords.numpy(), k=min(5,BAG_SIZE))
        src, tgt = [], []
        for ni, nb in enumerate(nn):
            for nj in nb[1:]: src+=[ni,int(nj)]; tgt+=[int(nj),ni]
        graphs.append(Data(x=feats, coords=coords,
                           edge_index=torch.tensor([src,tgt],dtype=torch.long),
                           y=torch.tensor([lbl],dtype=torch.long)))
    return graphs

print('Building bags (takes ~3-5 min)...')
# PatchCamelyon only has 'train' and 'test' splits, so split train manually
ALL_TRAIN_RAW = make_graphs('train', 1000)
TEST_G = make_graphs('test', 200)

# Split training pool: 60% train, 20% val
rng = np.random.default_rng(42)
idxs = rng.permutation(len(ALL_TRAIN_RAW))
n_tr = int(0.6 * len(ALL_TRAIN_RAW))
n_va = int(0.2 * len(ALL_TRAIN_RAW))

ALL_TRAIN = [ALL_TRAIN_RAW[i] for i in idxs[:n_tr]]
VAL_G = [ALL_TRAIN_RAW[i] for i in idxs[n_tr:n_tr+n_va]]

print(f'All bags: train={len(ALL_TRAIN)}  val={len(VAL_G)}  test={len(TEST_G)}')

Loading PatchCamelyon...
Building bags (takes ~3-5 min)...
All bags: train=600  val=200  test=200


## Cell 5 — Create data loaders
print('Creating data loaders...')
tr_ldr  = PyGLoader(ALL_TRAIN, batch_size=8, shuffle=True)
va_ldr  = PyGLoader(VAL_G,     batch_size=8, shuffle=False)
te_ldr  = PyGLoader(TEST_G,    batch_size=8, shuffle=False)
print(f'Loaders ready: {len(tr_ldr)} train, {len(va_ldr)} val, {len(te_ldr)} test batches')

In [ ]:
class VQCEncoder(nn.Module):
    """
    Fast VQC encoder — processes patches in parallel using batch operations.
    Key optimisation: reduce n_qubits to 2 (4 amplitudes instead of 8)
    and use lightning.qubit if available for 10x speed boost.
    """
    def __init__(self, in_dim=512, n_qubits=2, n_layers=1):
        super().__init__()
        self.n_qubits = n_qubits
        self.vqc_dim  = 2 ** n_qubits   # 4 for 2 qubits

        self.proj = nn.Sequential(
            nn.Linear(in_dim, self.vqc_dim),
            nn.Tanh(),
        )

        # Use lightning.qubit if available (10x faster than default.qubit)
        try:
            dev = qml.device('lightning.qubit', wires=n_qubits)
            print('Using lightning.qubit (fast)')
        except Exception:
            dev = qml.device('default.qubit', wires=n_qubits)
            print('Using default.qubit (install pennylane-lightning for 10x speedup)')

        @qml.qnode(dev, interface='torch', diff_method='parameter-shift')
        def circuit(inputs, weights):
            qml.AmplitudeEmbedding(inputs, wires=range(n_qubits), normalize=True)
            for l in range(n_layers):
                for q in range(n_qubits):
                    qml.RY(weights[l, 0, q], wires=q)
                    qml.RZ(weights[l, 1, q], wires=q)
                qml.CNOT(wires=[0, 1] if n_qubits >= 2 else [0, 0])
            return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]

        self.vqc = qml.qnn.TorchLayer(circuit, {'weights': (n_layers, 2, n_qubits)})
        self.out_dim = self.vqc_dim + n_qubits  # 4 + 2 = 6

    def forward(self, x):
        p    = self.proj(x)
        p_n  = F.normalize(p, p=2, dim=1)
        q_out = self.vqc(p_n)
        return torch.cat([p, q_out], dim=1)

In [ ]:
RESULTS = {}  # store all experiment results

tr_ldr  = PyGLoader(ALL_TRAIN, batch_size=8, shuffle=True)
va_ldr  = PyGLoader(VAL_G,     batch_size=8, shuffle=False)
te_ldr  = PyGLoader(TEST_G,    batch_size=8, shuffle=False)

print('EXPERIMENT 1: Classical vs Quantum — full training data')
print('='*60)

# Classical baseline (no VQC)
print('\n--- Classical GNN (no VQC) ---')
model_c = PATHQHybrid(use_vqc=False).to(DEVICE)
res_c, hist_c = run_experiment(
    model_c, tr_ldr, va_ldr, te_ldr, DEVICE,
    label='classical_full', epochs=40,
    ckpt=str(CKPT_DIR/'w5_classical.pth')
)
RESULTS['classical_full'] = res_c

# Quantum hybrid (with VQC)
print('\n--- Quantum Hybrid (VQC + GNN) ---')
print('Note: VQC forward passes are slower than classical. Be patient.')
model_q = PATHQHybrid(use_vqc=True, n_qubits=3, n_layers=2).to(DEVICE)
res_q, hist_q = run_experiment(
    model_q, tr_ldr, va_ldr, te_ldr, DEVICE,
    label='quantum_full', epochs=40,
    ckpt=str(CKPT_DIR/'w5_quantum.pth')
)
RESULTS['quantum_full'] = res_q

print('\n=== FULL DATA COMPARISON ===')
print(f'  Classical AUC: {res_c["auc"]:.4f}')
print(f'  Quantum AUC:   {res_q["auc"]:.4f}')
delta = res_q['auc'] - res_c['auc']
print(f'  Delta:        {delta:+.4f}  ({"quantum wins" if delta > 0 else "classical wins — expected with random init"})')

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

EXPERIMENT 1: Classical vs Quantum — full training data

--- Classical GNN (no VQC) ---


wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:

RESULTS = {}  # store all experiment results

print('EXPERIMENT 1: Classical vs Quantum — full training data')
print('='*60)

# Classical baseline (no VQC)
print('\n--- Classical GNN (no VQC) ---')
model_c = PATHQHybrid(use_vqc=False).to(DEVICE)
res_c, hist_c = run_experiment(
    model_c, tr_ldr, va_ldr, te_ldr, DEVICE,
    label='classical_full', epochs=40,
    ckpt=str(CKPT_DIR/'w5_classical.pth')
)
RESULTS['classical_full'] = res_c

# Quantum hybrid (with VQC)
print('\n--- Quantum Hybrid (VQC + GNN) ---')
print('Note: VQC forward passes are slower than classical. Be patient.')
model_q = PATHQHybrid(use_vqc=True, n_qubits=3, n_layers=2).to(DEVICE)
res_q, hist_q = run_experiment(
    model_q, tr_ldr, va_ldr, te_ldr, DEVICE,
    label='quantum_full', epochs=40,
    ckpt=str(CKPT_DIR/'w5_quantum.pth')
)
RESULTS['quantum_full'] = res_q

print('\n=== FULL DATA COMPARISON ===')
print(f'  Classical AUC: {res_c["auc"]:.4f}')
print(f'  Quantum AUC:   {res_q["auc"]:.4f}')
delta = res_q['auc'] - res_c['auc']
print(f'  Delta:        {delta:+.4f}  ({"quantum wins" if delta > 0 else "classical wins — expected with random init"})')

In [ ]:
print('EXPERIMENT 2: VQC Layer Ablation')
print('='*50)
print('How many VQC layers is optimal?')
print()

ablation_results = {}
for n_layers in [1, 2, 3]:
    print(f'--- {n_layers}-layer VQC ---')
    m = PATHQHybrid(use_vqc=True, n_qubits=3, n_layers=n_layers).to(DEVICE)
    res, _ = run_experiment(
        m, tr_ldr, va_ldr, te_ldr, DEVICE,
        label=f'vqc_layers_{n_layers}', epochs=30,
        ckpt=str(CKPT_DIR/f'w5_vqc_l{n_layers}.pth'),
        use_wandb=True
    )
    ablation_results[n_layers] = res

print('\n=== LAYER ABLATION RESULTS ===')
print(f'  {"Layers":>8}  {"AUC":>8}  {"F1":>8}  {"Sensitivity":>12}')
for nl, r in ablation_results.items():
    print(f'  {nl:>8}  {r["auc"]:>8.4f}  {r["f1"]:>8.4f}  {r["sensitivity"]:>12.4f}')

RESULTS['ablation_layers'] = ablation_results

## Cell 7 — Experiment 3: Low-Data Regime (Quantum Advantage)

In [ ]:
print('EXPERIMENT 3: Low-Data Regime')
print('='*55)
print('Hypothesis: quantum kernels generalise better with fewer labels.')
print('Expected: quantum advantage grows as training data shrinks.')
print()

low_data_results = {}

for pct in [0.20, 0.50, 1.00]:
    n_use = max(10, int(len(ALL_TRAIN) * pct))
    subset_idx = np.random.choice(len(ALL_TRAIN), n_use, replace=False)
    subset     = [ALL_TRAIN[i] for i in subset_idx]
    sub_ldr    = PyGLoader(subset, batch_size=8, shuffle=True)

    print(f'--- {int(pct*100)}% of training data ({n_use} bags) ---')

    # Classical
    mc = PATHQHybrid(use_vqc=False).to(DEVICE)
    rc, _ = run_experiment(mc, sub_ldr, va_ldr, te_ldr, DEVICE,
                           label=f'c_{int(pct*100)}pct', epochs=30,
                           ckpt=str(CKPT_DIR/f'w5_c_{int(pct*100)}.pth'))

    # Quantum
    mq = PATHQHybrid(use_vqc=True, n_qubits=3, n_layers=2).to(DEVICE)
    rq, _ = run_experiment(mq, sub_ldr, va_ldr, te_ldr, DEVICE,
                           label=f'q_{int(pct*100)}pct', epochs=30,
                           ckpt=str(CKPT_DIR/f'w5_q_{int(pct*100)}.pth'))

    low_data_results[pct] = {'classical': rc, 'quantum': rq,
                              'delta': rq['auc'] - rc['auc']}
    print(f'  Classical: {rc["auc"]:.4f}  Quantum: {rq["auc"]:.4f}  '
          f'Delta: {rq["auc"]-rc["auc"]:+.4f}')
    print()

RESULTS['low_data'] = low_data_results
print('\n=== LOW-DATA SUMMARY ===')
print(f'  {"Data %":>8}  {"Classical":>10}  {"Quantum":>10}  {"Delta":>8}')
for pct, r in low_data_results.items():
    print(f'  {int(pct*100):>7}%  {r["classical"]["auc"]:>10.4f}  '
          f'{r["quantum"]["auc"]:>10.4f}  {r["delta"]:>+8.4f}')
print()
print('If delta is positive at 20% but negative at 100%,')
print('that is the quantum advantage in the low-data regime.')
print('This is your strongest research claim.')

## Cell 8 — Final Comparison Figure (Paper Ready)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('PATHQ Experimental Results', fontsize=14, fontweight='bold')

# ── Plot 1: Classical vs Quantum full data ────────────────────────────────
models  = ['Classical\n(No VQC)', 'Quantum\n(VQC+GNN)']
aucs    = [RESULTS['classical_full']['auc'], RESULTS['quantum_full']['auc']]
colors  = ['#888780', '#0A6B6B']
bars    = axes[0].bar(models, aucs, color=colors, alpha=0.85, edgecolor='white', width=0.5)
axes[0].set_ylim(0.5, 1.0)
axes[0].set_title('Full Data: Classical vs Quantum', fontsize=11)
axes[0].set_ylabel('AUC-ROC')
axes[0].axhline(0.85, color='red', ls='--', alpha=0.4, label='Target 0.85')
for bar, auc in zip(bars, aucs):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{auc:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# ── Plot 2: Layer ablation ────────────────────────────────────────────────
abl     = RESULTS.get('ablation_layers', {})
if abl:
    layers  = list(abl.keys())
    abl_auc = [abl[l]['auc'] for l in layers]
    axes[1].plot(layers, abl_auc, 'o-', color='#534AB7', lw=2, ms=10)
    for x, y in zip(layers, abl_auc):
        axes[1].annotate(f'{y:.3f}', (x,y), textcoords='offset points',
                         xytext=(5,5), fontsize=9)
axes[1].set_title('VQC Layer Ablation', fontsize=11)
axes[1].set_xlabel('Number of VQC Layers'); axes[1].set_ylabel('Test AUC-ROC')
axes[1].set_xticks([1,2,3]); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0.5, 1.0)

# ── Plot 3: Low-data regime ────────────────────────────────────────────────
ld = RESULTS.get('low_data', {})
if ld:
    pcts  = [int(p*100) for p in ld.keys()]
    c_auc = [ld[p]['classical']['auc'] for p in ld.keys()]
    q_auc = [ld[p]['quantum']['auc']   for p in ld.keys()]
    axes[2].plot(pcts, c_auc, 's--', color='#888780', lw=2, ms=8, label='Classical')
    axes[2].plot(pcts, q_auc, 'o-',  color='#0A6B6B', lw=2, ms=8, label='Quantum')
    axes[2].fill_between(pcts, c_auc, q_auc, alpha=0.15, color='#0A6B6B')
axes[2].set_title('Low-Data Regime', fontsize=11)
axes[2].set_xlabel('Training Data %'); axes[2].set_ylabel('Test AUC-ROC')
axes[2].set_xticks([20,50,100]); axes[2].grid(alpha=0.3)
axes[2].set_ylim(0.5, 1.0); axes[2].legend()

plt.tight_layout()
plt.savefig(str(OUT_DIR/'week5_all_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Paper figure saved: outputs/week5_all_results.png')

# Save all results
def to_serialisable(d):
    if isinstance(d, dict):
        return {str(k): to_serialisable(v) for k,v in d.items()}
    if isinstance(d, (np.float32, np.float64)): return float(d)
    if isinstance(d, np.ndarray): return d.tolist()
    return d

with open(OUT_DIR/'week5_results.json','w') as f:
    json.dump(to_serialisable(RESULTS), f, indent=2)
print('Results saved: outputs/week5_results.json')

In [ ]:
# Load baseline from Week 3 for full comparison
try:
    with open(OUT_DIR/'baseline_results.json') as f:
        baseline = json.load(f)
    print('=== COMPLETE RESULTS SUMMARY ===')
    print(f'  Week 3 Classical AUC (PCam): {baseline.get("pcam_classical_auc", "N/A")}')
    print(f'  Week 5 Classical AUC (PCam): {RESULTS["classical_full"]["auc"]:.4f}')
    print(f'  Week 5 Quantum AUC   (PCam): {RESULTS["quantum_full"]["auc"]:.4f}')
except FileNotFoundError:
    print('Run Week 3 first to generate baseline_results.json')

print()
print('WEEK 5 COMPLETE')
print('='*50)
print('Next: week6_xai_complete.ipynb')
print('You will implement all three XAI layers and generate')
print('the final Grad-CAM + attention + quantum circuit figures.')